# Marketing Campaign Analytics - Exploratory Data Analysis (EDA)
## Notebook 02: Python Visual & Empirical Exploration

**Analytical Purpose:**
This notebook conducts an exhaustive exploratory data analysis on the two cleaned datasets:
1. `campaigns_clean.csv` (200,000 ad campaign records)
2. `customers_clean.csv` (2,236 customer profile records)

**Crucial Architectural Principle:** The two datasets share NO identifier or join key.
They represent two separate analytical perspectives (macro campaign performance vs. micro customer behavior)
and are analyzed independently throughout.

In [ ]:
import os
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Safe display helper for both notebook interactive and standalone script runs
try:
    from IPython.display import display
except ImportError:
    display = print

# Configure Matplotlib & Seaborn aesthetics
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid', palette='crest')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

# Set reproducible random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Resolve relative file paths
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = BASE_DIR / 'data' / 'cleaned'
FIGURES_DIR = BASE_DIR / 'reports' / 'figures'
STAT_DIR = BASE_DIR / 'reports' / 'stat_checks'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
STAT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base Directory   : {BASE_DIR.resolve()}")
print(f"Cleaned Data Dir : {DATA_DIR.resolve()}")
print(f"Figures Directory: {FIGURES_DIR.resolve()}")

# Load Cleaned DataFrames from CSV
df_camp = pd.read_csv(DATA_DIR / 'campaigns_clean.csv')
df_cust = pd.read_csv(DATA_DIR / 'customers_clean.csv')

# Ensure campaign date is datetime
df_camp['campaign_date'] = pd.to_datetime(df_camp['campaign_date'])

print(f"Loaded campaigns_clean.csv : {df_camp.shape[0]:,} rows x {df_camp.shape[1]} columns")
print(f"Loaded customers_clean.csv : {df_cust.shape[0]:,} rows x {df_cust.shape[1]} columns")

# Helper function to save figures consistently
def save_chart(fig, filename):
    filepath = FIGURES_DIR / filename
    fig.savefig(filepath, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved figure: {filepath.name}")

---
## Section 1: Setup and Data Overview
**Business Question:** What are the genuine structural characteristics, data types, missing value rates, date boundaries, and channel distributions of the datasets?

In [ ]:
camp_n_rows, camp_n_cols = df_camp.shape
cust_n_rows, cust_n_cols = df_cust.shape
camp_nulls = df_camp.isnull().sum().sum()
cust_nulls = df_cust.isnull().sum().sum()

min_date = df_camp['campaign_date'].min().strftime('%Y-%m-%d')
max_date = df_camp['campaign_date'].max().strftime('%Y-%m-%d')
channel_counts = df_camp['channel_used'].value_counts()

min_roas = df_camp['roas'].min()
max_roas = df_camp['roas'].max()
min_roi = df_camp['roi'].min()
max_roi = df_camp['roi'].max()

print("--- Campaign Data Types & Overview ---")
print(df_camp.dtypes)
print(f"\nMissing values in campaigns: {camp_nulls}")
print(f"Missing values in customers: {cust_nulls}")
print(f"Campaign Date Range: {min_date} to {max_date}")
print("\nCampaign Count per Channel:")
print(channel_counts)

**Section 1 Takeaway (Data Overview):**

In [ ]:
takeaway_s1 = (
    f"Takeaway: The campaigns dataset contains {camp_n_rows:,} records spanning {min_date} to {max_date} "
    f"with 0 missing values, uniformly distributed across 6 channels (~{channel_counts.mean():,.0f} campaigns each). "
    f"The data is synthetic: ROI is bounded strictly in [{min_roi:.1f}, {max_roi:.1f}], which forces ROAS to fall "
    f"strictly between {min_roas:.1f} and {max_roas:.1f} (ROAS = 1 + ROI by construction), meaning 0 campaigns lose money."
)
print(takeaway_s1)

---
## Section 2: Numerical Distributions & Ceiling Constraints
**Business Question:** How are the continuous performance and financial metrics distributed, and do artificial boundaries or capping limits exist?

In [ ]:
numeric_cols = [
    'acquisition_cost', 'revenue', 'profit', 
    'conversion_rate', 'clicks', 'impressions', 'roas'
]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

ceilings_bounds = {
    'conversion_rate': {'min': 0.01, 'max': 0.15, 'label': 'Cap: 0.15'},
    'clicks': {'min': 100, 'max': 1000, 'label': 'Cap: 1,000'},
    'impressions': {'min': 1000, 'max': 10000, 'label': 'Bounds: [1k, 10k]'},
    'roas': {'min': 3.0, 'max': 9.0, 'label': 'Bounds: [3.0, 9.0]'}
}

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    sns.histplot(df_camp[col], bins=30, ax=ax, color='#1f77b4', edgecolor='white', kde=False)
    ax.set_title(f"Distribution of {col.replace('_', ' ').title()}", fontsize=11, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel("Campaign Count")
    
    if col in ceilings_bounds:
        b_info = ceilings_bounds[col]
        ax.axvline(b_info['max'], color='#d62728', linestyle='--', linewidth=1.5, label=f"Max ({b_info['max']})")
        ax.axvline(b_info['min'], color='#2ca02c', linestyle=':', linewidth=1.5, label=f"Min ({b_info['min']})")
        ax.legend(loc='upper right', fontsize=8)

# Hide unused subplots
for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

fig.suptitle("Section 2: Key Campaign Metric Distributions & Synthetic Bounds", fontsize=14, fontweight='bold', y=0.99)
fig.tight_layout()
save_chart(fig, '01_campaign_numeric_distributions.png')

# Compute ceiling metrics
ceiling_clicks_conv = ((df_camp['clicks'] == 1000) & (df_camp['conversion_rate'] == 0.15)).sum()
conv_at_max = (df_camp['conversion_rate'] == 0.15).sum()
clicks_at_max = (df_camp['clicks'] == 1000).sum()

**Section 2 Takeaway (Distributions & Constraints):**

In [ ]:
takeaway_s2 = (
    f"Takeaway: Every core metric follows a flat uniform distribution characteristic of synthetic generation. "
    f"Hard artificial bounds constrain conversion rate to [0.01, 0.15] ({conv_at_max:,} campaigns at cap) "
    f"and clicks to [100, 1,000] ({clicks_at_max:,} campaigns at cap), with exactly {ceiling_clicks_conv} campaigns "
    f"coinciding at both ceilings. Financial metrics (revenue and profit) reflect these bounded multiplicative products."
)
print(takeaway_s2)

---
## Section 3: Channel Performance & Estimated Unit Economics
**Business Question:** How do marketing channels compare on aggregate ROAS, ROI %, and estimated Customer Acquisition Cost (CAC)?

In [ ]:
df_camp['estimated_conversions'] = df_camp['clicks'] * df_camp['conversion_rate']

channel_grp = df_camp.groupby('channel_used')
spend_by_ch = channel_grp['acquisition_cost'].sum()
rev_by_ch = channel_grp['revenue'].sum()
profit_by_ch = channel_grp['profit'].sum()
clicks_by_ch = channel_grp['clicks'].sum()
conv_by_ch = channel_grp['estimated_conversions'].sum()

roas_by_ch = rev_by_ch / spend_by_ch
roi_pct_by_ch = (profit_by_ch / spend_by_ch) * 100
cac_by_ch = spend_by_ch / conv_by_ch
cpc_by_ch = spend_by_ch / clicks_by_ch

df_channel_perf = pd.DataFrame({
    'channel': roas_by_ch.index,
    'roas': roas_by_ch.values,
    'roi_pct': roi_pct_by_ch.values,
    'cac': cac_by_ch.values,
    'cpc': cpc_by_ch.values
}).sort_values('roas', ascending=False)

# Chart 3A: Baseline Full-Scale Channel Scorecard (Y-Axis Starting at Zero)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.5))

sns.barplot(data=df_channel_perf, x='channel', y='roas', ax=ax1, palette='crest')
ax1.set_title("Aggregate ROAS by Channel (Zero-Baseline)", fontweight='bold')
ax1.set_ylim(0, 7.5)
ax1.set_ylabel("ROAS (Revenue / Spend)")
ax1.set_xlabel("Channel")
for p in ax1.patches:
    ax1.annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                 ha='center', va='center', color='white', fontweight='bold', fontsize=9)

sns.barplot(data=df_channel_perf, x='channel', y='roi_pct', ax=ax2, palette='crest')
ax2.set_title("Aggregate ROI % by Channel (Zero-Baseline)", fontweight='bold')
ax2.set_ylim(0, 650)
ax2.set_ylabel("ROI (%)")
ax2.set_xlabel("Channel")
for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                 ha='center', va='center', color='white', fontweight='bold', fontsize=9)

sns.barplot(data=df_channel_perf, x='channel', y='cac', ax=ax3, palette='crest')
ax3.set_title("Estimated CAC by Channel (Zero-Baseline)", fontweight='bold')
ax3.set_ylim(0, 350)
ax3.set_ylabel("Estimated CAC ($)")
ax3.set_xlabel("Channel")
for p in ax3.patches:
    ax3.annotate(f"${p.get_height():.1f}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                 ha='center', va='center', color='white', fontweight='bold', fontsize=9)

fig.suptitle("Section 3A: Channel Macro Performance Scorecard (Un-truncated Axes)", fontsize=13, fontweight='bold')
fig.tight_layout()
save_chart(fig, '02_channel_performance_scorecard.png')

# Chart 3B: Zoomed ROAS View with 95% Confidence Intervals
fig, ax = plt.subplots(figsize=(9, 5))
# Calculate mean and 95% CI per channel
channel_stats = df_camp.groupby('channel_used')['roas'].agg(['mean', 'sem']).reset_index()
channel_stats['ci_lower'] = channel_stats['mean'] - 1.96 * channel_stats['sem']
channel_stats['ci_upper'] = channel_stats['mean'] + 1.96 * channel_stats['sem']
channel_stats = channel_stats.sort_values('mean', ascending=False)

ax.errorbar(
    channel_stats['channel_used'], channel_stats['mean'],
    yerr=1.96 * channel_stats['sem'], fmt='o', color='#1f77b4',
    ecolor='#d62728', elinewidth=2, capsize=5, markersize=7
)
ax.set_ylim(5.95, 6.06)
ax.set_title("Zoomed View: Channel ROAS with 95% Confidence Intervals\n(Note: Y-axis zoomed to [5.95, 6.06] to show CI overlap)", 
             fontsize=11, fontweight='bold')
ax.set_ylabel("Mean ROAS (95% CI)")
ax.set_xlabel("Marketing Channel")
for i, row in channel_stats.iterrows():
    ax.annotate(f"{row['mean']:.4f}\n[{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]",
                (row['channel_used'], row['ci_upper'] + 0.003), ha='center', fontsize=8)

fig.tight_layout()
save_chart(fig, '03_roas_by_channel.png')

min_ch_roas = df_channel_perf['roas'].min()
max_ch_roas = df_channel_perf['roas'].max()
spread_ch_roas = max_ch_roas - min_ch_roas
spread_pct_roas = (spread_ch_roas / min_ch_roas) * 100
min_cac = df_channel_perf['cac'].min()
max_cac = df_channel_perf['cac'].max()

**Section 3 Takeaway (Channel Performance):**

In [ ]:
takeaway_s3 = (
    f"Takeaway: On zero-baseline charts, performance across all 6 channels appears indistinguishable. "
    f"Aggregate channel ROAS spans from {min_ch_roas:.4f} (Instagram) to {max_ch_roas:.4f} (Facebook), "
    f"a practically negligible spread of {spread_ch_roas:.4f} ({spread_pct_roas:.2f}%). "
    f"Estimated CAC spans a tight range of ${min_cac:.2f} (Website) to ${max_cac:.2f} (Google Ads), "
    f"and 95% confidence intervals heavily overlap around the grand mean (~6.00)."
)
print(takeaway_s3)

---
## Section 4: Variation Inside Channels vs. Between Channels
**Business Question:** Does performance vary more between different channels or within each individual channel?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df_camp, x='channel_used', y='roas', ax=ax, palette='crest', fliersize=1)
ax.set_title("Section 4: Distribution of ROAS Within Each Channel (Boxplot)", fontsize=12, fontweight='bold')
ax.set_xlabel("Channel Used")
ax.set_ylabel("ROAS")
ax.set_ylim(2.5, 9.5)

# Annotate median and IQR
medians = df_camp.groupby('channel_used')['roas'].median()
iqrs = df_camp.groupby('channel_used')['roas'].apply(lambda x: x.quantile(0.75) - x.quantile(0.25))
for idx, ch in enumerate(medians.index):
    ax.annotate(f"Median: {medians[ch]:.2f}\nIQR: {iqrs[ch]:.2f}", 
                (idx, 8.3), ha='center', fontsize=8, bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

fig.tight_layout()
save_chart(fig, '04_roas_distribution_by_channel_boxplot.png')

avg_within_std = df_camp.groupby('channel_used')['roas'].std().mean()
between_mean_std = df_camp.groupby('channel_used')['roas'].mean().std()

**Section 4 Takeaway (Within vs. Between Variation):**

In [ ]:
takeaway_s4 = (
    f"Takeaway: Within-channel variation (standard deviation ~{avg_within_std:.2f}, spanning from 3.0 to 9.0) "
    f"is vastly larger than between-channel variation (standard deviation across channel means is only {between_mean_std:.4f}). "
    f"Individual campaign attributes within a channel drive returns far more than the choice of channel itself."
)
print(takeaway_s4)

---
## Section 5: Other Campaign Attributes & Dimension Effect Sizes
**Business Question:** Do non-channel attributes (format, audience, segment, geography, language, duration) create meaningful return differences?

In [ ]:
other_dims = ['campaign_type', 'target_audience', 'customer_segment', 'location', 'language', 'duration_days']

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
axes = axes.flatten()

for i, dim in enumerate(other_dims):
    ax = axes[i]
    dim_stats = df_camp.groupby(dim)['roas'].agg(['mean', 'sem']).reset_index()
    sns.barplot(data=dim_stats, x=dim, y='mean', ax=ax, palette='crest')
    ax.errorbar(
        range(len(dim_stats)), dim_stats['mean'],
        yerr=1.96 * dim_stats['sem'], fmt='none', color='black', capsize=3
    )
    ax.set_title(f"ROAS by {dim.replace('_', ' ').title()}", fontweight='bold', fontsize=10)
    ax.set_ylabel("Mean ROAS")
    ax.set_xlabel(dim.replace('_', ' ').title())
    ax.set_ylim(0, 7.5)
    ax.tick_params(axis='x', rotation=25)

fig.suptitle("Section 5A: Mean ROAS Across Other Campaign Attributes (Zero-Baseline)", fontsize=13, fontweight='bold')
fig.tight_layout()
save_chart(fig, '05_roas_by_other_dimensions.png')

# Chart 5B: Channel x Campaign Type Heatmap of ROAS with Sample Sizes
pivot_roas = df_camp.pivot_table(index='channel_used', columns='campaign_type', values='roas', aggfunc='mean')
pivot_counts = df_camp.pivot_table(index='channel_used', columns='campaign_type', values='roas', aggfunc='count')

annot_labels = np.empty(pivot_roas.shape, dtype=object)
for r in range(pivot_roas.shape[0]):
    for c in range(pivot_roas.shape[1]):
        annot_labels[r, c] = f"{pivot_roas.iloc[r, c]:.2f}\n(n={pivot_counts.iloc[r, c]:,})"

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot_roas, annot=annot_labels, fmt='', cmap='YlGnBu', cbar_kws={'label': 'Mean ROAS'}, ax=ax, vmin=5.8, vmax=6.2)
ax.set_title("Section 5B: ROAS by Channel x Campaign Type (Annotated with Cell Size)", fontsize=12, fontweight='bold')
ax.set_xlabel("Campaign Type")
ax.set_ylabel("Channel Used")
fig.tight_layout()
save_chart(fig, '06_channel_by_type_heatmap.png')

# Chart 5C: Rank Dimensions by Eta-Squared
# Read the previously computed eta-squared CSV or compute directly
all_dims = ['channel_used', 'campaign_type', 'target_audience', 'customer_segment', 'location', 'language', 'duration_days']
grand_m = df_camp['roas'].mean()
ss_t = ((df_camp['roas'] - grand_m) ** 2).sum()
eta_list = []
for d in all_dims:
    grp = df_camp.groupby(d)['roas']
    ss_b = (grp.count() * ((grp.mean() - grand_m) ** 2)).sum()
    eta_list.append({'dimension': d, 'eta_squared': ss_b / ss_t, 'pct_var': (ss_b / ss_t) * 100})

df_eta = pd.DataFrame(eta_list).sort_values('eta_squared', ascending=True)

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(df_eta['dimension'], df_eta['pct_var'], color='#2b83ba')
ax.set_title("Section 5C: Dimension Ranking by Effect Size (% of ROAS Variance Explained, η²)", fontsize=11, fontweight='bold')
ax.set_xlabel("Percentage of ROAS Variance Explained (%)")
ax.set_xlim(0, 0.007)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.0002, bar.get_y() + bar.get_height()/2, f"{w:.6f}%", va='center', fontsize=8, fontweight='bold')

fig.tight_layout()
save_chart(fig, '07_eta_squared_ranking.png')

top_dim = df_eta.iloc[-1]
bot_dim = df_eta.iloc[0]

**Section 5 Takeaway (Other Attributes & Effect Sizes):**

In [ ]:
takeaway_s5 = (
    f"Takeaway: Across all 7 tested dimensions, ROAS remains virtually identical (~6.00). "
    f"The top explanatory dimension is {top_dim['dimension']} (explaining merely {top_dim['pct_var']:.6f}% of variance), "
    f"while {bot_dim['dimension']} explains only {bot_dim['pct_var']:.6f}%. "
    f"No single categorical dimension provides meaningful predictive leverage over campaign returns."
)
print(takeaway_s5)

---
## Section 6: Time Trends, Seasonality, and Pacing
**Business Question:** Do marketing spend, revenue, and profit display organic seasonality or trends over time?

In [ ]:
df_camp['year_month'] = df_camp['campaign_date'].dt.to_period('M').dt.to_timestamp()
monthly_agg = df_camp.groupby('year_month').agg({
    'acquisition_cost': 'sum',
    'revenue': 'sum',
    'profit': 'sum'
}).reset_index()

monthly_agg['mom_revenue_growth_pct'] = monthly_agg['revenue'].pct_change() * 100

# Chart 6A: Monthly Spend, Revenue, Profit
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(monthly_agg['year_month'], monthly_agg['revenue'] / 1e6, marker='o', color='#2ca02c', label='Revenue ($M)')
ax1.plot(monthly_agg['year_month'], monthly_agg['profit'] / 1e6, marker='s', color='#1f77b4', label='Profit ($M)')
ax1.plot(monthly_agg['year_month'], monthly_agg['acquisition_cost'] / 1e6, marker='^', color='#d62728', label='Spend ($M)')
ax1.set_ylabel("Total Financials ($ Millions)")
ax1.set_title("Monthly Spend, Revenue, and Profit Across 2021", fontweight='bold')
ax1.legend(loc='lower right')

# Chart 6B: MoM Growth %
ax2.bar(monthly_agg['year_month'], monthly_agg['mom_revenue_growth_pct'], width=20, color='#7b3294')
ax2.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax2.set_ylabel("MoM Revenue Growth (%)")
ax2.set_title("Month-over-Month Revenue Growth Rate (%)", fontweight='bold')
ax2.set_xlabel("Date")

fig.suptitle("Section 6A: Financial Pacing and Monthly Volatility", fontsize=13, fontweight='bold')
fig.tight_layout()
save_chart(fig, '08_monthly_financial_trends.png')

# Chart 6C: Quarterly Profit by Channel
quarterly_perf = df_camp.groupby(['year', 'quarter', 'channel_used'])['profit'].sum().reset_index()
quarterly_perf['quarter_lbl'] = 'Q' + quarterly_perf['quarter'].astype(str)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=quarterly_perf, x='quarter_lbl', y='profit', hue='channel_used', ax=ax, palette='crest')
ax.set_title("Quarterly Profit by Channel Across 2021", fontsize=12, fontweight='bold')
ax.set_ylabel("Total Profit ($)")
ax.set_xlabel("Calendar Quarter")
ax.legend(title="Channel", bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
save_chart(fig, '09_quarterly_channel_profit.png')

feb_growth = monthly_agg.loc[monthly_agg['year_month'] == '2021-02-01', 'mom_revenue_growth_pct'].values[0]
mar_growth = monthly_agg.loc[monthly_agg['year_month'] == '2021-03-01', 'mom_revenue_growth_pct'].values[0]

**Section 6 Takeaway (Time Trends):**

In [ ]:
takeaway_s6 = (
    f"Takeaway: There is zero organic seasonality or temporal trend. Monthly revenue fluctuates strictly with "
    f"calendar month length: February dips by {feb_growth:.2f}% (28 days) and March rebounds by +{mar_growth:.2f}% (31 days). "
    f"Daily revenue pacing is virtually constant throughout the entire annual timeline."
)
print(takeaway_s6)

---
## Section 7: Spend vs. Revenue & Diminishing Returns Evaluation
**Business Question:** Do returns diminish as acquisition cost scales up, or does the relationship remain strictly linear?

In [ ]:
df_sample = df_camp.sample(n=10000, random_state=RANDOM_SEED)

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=df_sample, x='acquisition_cost', y='revenue', 
    hue='channel_used', alpha=0.35, s=20, ax=ax, palette='crest'
)
# Add linear fit
m, b = np.polyfit(df_sample['acquisition_cost'], df_sample['revenue'], 1)
x_vals = np.linspace(df_sample['acquisition_cost'].min(), df_sample['acquisition_cost'].max(), 100)
ax.plot(x_vals, m * x_vals + b, color='#d62728', linestyle='--', linewidth=2, label=f"Linear Fit (Slope: {m:.2f})")

ax.set_title("Section 7: Acquisition Cost vs. Revenue (10,000 Campaign Sample)", fontsize=12, fontweight='bold')
ax.set_xlabel("Acquisition Cost ($)")
ax.set_ylabel("Revenue ($)")
ax.legend(title="Channel", bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
save_chart(fig, '10_spend_vs_revenue_scatter.png')

# Evaluate diminishing returns: correlation between cost and ROAS
cost_roas_corr = df_camp['acquisition_cost'].corr(df_camp['roas'])

**Section 7 Takeaway (Diminishing Returns):**

In [ ]:
takeaway_s7 = (
    f"Takeaway: Returns do not diminish at higher budget levels. The relationship between spend and revenue "
    f"is strictly linear with an average slope of {m:.2f} (r = {df_camp['acquisition_cost'].corr(df_camp['revenue']):.2f}). "
    f"The correlation between acquisition cost and ROAS is r = {cost_roas_corr:.4f}, demonstrating no scale penalty."
)
print(takeaway_s7)

---
## Section 8: Correlation Matrix of Numeric Metrics
**Business Question:** What linear correlations exist between numerical attributes?
*Important Note:* Revenue, profit, ROI, and ROAS are mechanically linked by construction (`Revenue = Cost * (1 + ROI)`).

In [ ]:
corr_fields = [
    'acquisition_cost', 'revenue', 'profit', 'conversion_rate', 
    'roi', 'roas', 'clicks', 'impressions', 'ctr', 'cpc', 'duration_days'
]
df_corr = df_camp[corr_fields].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax, cbar_kws={'label': 'Pearson Correlation'})
ax.set_title("Section 8: Correlation Matrix of Campaign Numerical Features", fontsize=12, fontweight='bold')
fig.tight_layout()
save_chart(fig, '11_correlation_heatmap.png')

**Section 8 Takeaway (Correlation Matrix):**

In [ ]:
takeaway_s8 = (
    f"Takeaway: High correlations exist exclusively where variables are linked by definition: "
    f"revenue and profit (r = {df_corr.loc['revenue', 'profit']:.2f}), and ROI and ROAS (r = {df_corr.loc['roi', 'roas']:.2f}). "
    f"Operational metrics (clicks, impressions, conversion rate, cost) show near-zero mutual correlation (r ~ 0.00)."
)
print(takeaway_s8)

---
## Section 9: Customer Behavioral Demographics & Segmentation
**Business Question:** How do customer purchasing habits and campaign acceptance associate with income tiers and age cohorts?
*Crucial Architecture Note:* Customer dataset is separate from campaign dataset; findings represent exploratory associations.

In [ ]:
# Chart 9A: Customer Cohort Distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

age_counts = df_cust['age_group'].value_counts()[['<30', '30-44', '45-59', '60+']]
sns.barplot(x=age_counts.index, y=age_counts.values, ax=ax1, palette='crest')
ax1.set_title("Customer Distribution by Age Group (n labelled)", fontweight='bold')
ax1.set_ylabel("Number of Customers")
ax1.set_xlabel("Age Group")
for p in ax1.patches:
    ax1.annotate(f"n = {int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height() + 15),
                 ha='center', fontsize=9, fontweight='bold')
ax1.set_ylim(0, age_counts.max() * 1.15)

inc_counts = df_cust['income_group'].value_counts()[['Low (<$35k)', 'Middle ($35k-$65k)', 'High ($65k-$100k)', 'Affluent (>$100k)']]
sns.barplot(x=inc_counts.index, y=inc_counts.values, ax=ax2, palette='crest')
ax2.set_title("Customer Distribution by Income Group (n labelled)", fontweight='bold')
ax2.set_ylabel("Number of Customers")
ax2.set_xlabel("Income Group")
for p in ax2.patches:
    ax2.annotate(f"n = {int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height() + 15),
                 ha='center', fontsize=9, fontweight='bold')
ax2.set_ylim(0, inc_counts.max() * 1.15)
ax2.tick_params(axis='x', rotation=15)

fig.suptitle("Section 9A: Customer Segment Sample Sizes", fontsize=13, fontweight='bold')
fig.tight_layout()
save_chart(fig, '12_customer_demographic_distributions.png')

# Chart 9B: Income Group Spend & Acceptance (Excluding Affluent n=12 with Footnote)
# Filter main tiers (n >= 100)
df_cust_main_inc = df_cust[df_cust['income_group'] != 'Affluent (>$100k)'].copy()
tier_order = ['Low (<$35k)', 'Middle ($35k-$65k)', 'High ($65k-$100k)']

inc_stats = df_cust_main_inc.groupby('income_group').agg(
    mean_spend=('total_spend', 'mean'),
    sem_spend=('total_spend', 'sem'),
    n=('total_spend', 'count'),
    succ=('campaigns_accepted', lambda x: (x > 0).sum())
).loc[tier_order]

inc_stats['acc_rate'] = (inc_stats['succ'] / inc_stats['n']) * 100

# Wilson CI for acceptance
def calc_wilson(s, n):
    z = 1.95996
    p = s / n
    d = 1.0 + (z**2 / n)
    c = (p + (z**2 / (2 * n))) / d
    m = (z / d) * math.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))
    return (c - m) * 100, (c + m) * 100

w_err = []
for idx, r in inc_stats.iterrows():
    low, high = calc_wilson(r['succ'], r['n'])
    w_err.append((r['acc_rate'] - low, high - r['acc_rate']))
w_err = np.array(w_err).T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Spend bar chart
ax1.bar(inc_stats.index, inc_stats['mean_spend'], yerr=1.96 * inc_stats['sem_spend'], capsize=5, color='#2ca02c', edgecolor='black')
ax1.set_title("Average Spend by Income Tier (95% CI)", fontweight='bold')
ax1.set_ylabel("Average Total Spend ($)")
for idx, (lbl, r) in enumerate(inc_stats.iterrows()):
    ax1.annotate(f"${r['mean_spend']:.1f}", (idx, r['mean_spend'] / 2), ha='center', color='white', fontweight='bold')

# Acceptance bar chart
ax2.bar(inc_stats.index, inc_stats['acc_rate'], yerr=w_err, capsize=5, color='#1f77b4', edgecolor='black')
ax2.set_title("Campaign Acceptance Rate by Income Tier (95% Wilson CI)", fontweight='bold')
ax2.set_ylabel("Acceptance Rate (%)")
for idx, (lbl, r) in enumerate(inc_stats.iterrows()):
    ax2.annotate(f"{r['acc_rate']:.1f}%", (idx, r['acc_rate'] / 2), ha='center', color='white', fontweight='bold')

fig.suptitle("Section 9B: Customer Purchasing Power and Promotional Response by Income Tier", fontsize=13, fontweight='bold')
fig.text(0.5, -0.02, "*Note: Affluent (>$100k) cohort excluded from main charts due to small sample size (n=12, avg spend $893.6, acceptance 33.3%).", 
         ha='center', fontsize=9, style='italic')
fig.tight_layout()
save_chart(fig, '13_customer_spend_acceptance_by_income.png')

# Chart 9C: Two-Way Heatmaps (Age x Income)
p_spend = df_cust.pivot_table(index='age_group', columns='income_group', values='total_spend', aggfunc='mean')[['Low (<$35k)', 'Middle ($35k-$65k)', 'High ($65k-$100k)', 'Affluent (>$100k)']].reindex(['<30', '30-44', '45-59', '60+'])
p_count = df_cust.pivot_table(index='age_group', columns='income_group', values='total_spend', aggfunc='count')[['Low (<$35k)', 'Middle ($35k-$65k)', 'High ($65k-$100k)', 'Affluent (>$100k)']].reindex(['<30', '30-44', '45-59', '60+']).fillna(0).astype(int)

annot_matrix = np.empty(p_spend.shape, dtype=object)
for r in range(p_spend.shape[0]):
    for c in range(p_spend.shape[1]):
        val = p_spend.iloc[r, c]
        cnt = p_count.iloc[r, c]
        annot_matrix[r, c] = f"${val:.0f}\n(n={cnt})" if not np.isnan(val) else f"N/A\n(n={cnt})"

fig, ax = plt.subplots(figsize=(9, 5.5))
sns.heatmap(p_spend, annot=annot_matrix, fmt='', cmap='YlGnBu', cbar_kws={'label': 'Average Spend ($)'}, ax=ax)
ax.set_title("Section 9C: Average Customer Spend by Age x Income (Cell n annotated)", fontsize=12, fontweight='bold')
ax.set_xlabel("Income Group")
ax.set_ylabel("Age Group (as of 2014)")
fig.tight_layout()
save_chart(fig, '14_customer_age_income_heatmaps.png')

high_inc_spend = inc_stats.loc['High ($65k-$100k)', 'mean_spend']
low_inc_spend = inc_stats.loc['Low (<$35k)', 'mean_spend']
spend_mult = high_inc_spend / low_inc_spend
high_inc_acc = inc_stats.loc['High ($65k-$100k)', 'acc_rate']
low_inc_acc = inc_stats.loc['Low (<$35k)', 'acc_rate']
acc_mult = high_inc_acc / low_inc_acc

**Section 9 Takeaway (Customer Demographics):**

In [ ]:
takeaway_s9 = (
    f"Takeaway: In the customer dataset, income shows a powerful association with purchasing behavior: "
    f"High-Income customers ($65k-$100k, n={inc_stats.loc['High ($65k-$100k)', 'n']}) average ${high_inc_spend:.2f} in spend "
    f"and {high_inc_acc:.1f}% acceptance, which is {spend_mult:.1f}x higher spend and {acc_mult:.1f}x higher acceptance "
    f"than Low-Income customers (<$35k, n={inc_stats.loc['Low (<$35k)', 'n']}). "
    f"When controlling for income tier, average spend within High Income is consistently ~$1,220-$1,430 across all ages, "
    f"confirming that the apparent age effect was largely a composition artifact of income."
)
print(takeaway_s9)

---
## Section 10: Key Business Takeaways Summary
Synthetic overview synthesized directly from computed data metrics.

In [ ]:
takeaway_summary = (
    "========================================================================================\n"
    "SUMMARY OF KEY PORTFOLIO FINDINGS (SYNTHESIZED FROM EMPIRICAL ANALYSIS)\n"
    "========================================================================================\n"
    f"1. MACRO CHANNEL PERFORMANCE IS UNIFORM (PRACTICALLY NEGLIGIBLE SPREAD):\n"
    f"   Across all 6 advertising channels (200,000 campaigns), aggregate ROAS spans from {min_ch_roas:.4f} (Instagram) "
    f"   to {max_ch_roas:.4f} (Facebook), a variance of only {spread_ch_roas:.4f} (0.67%).\n"
    f"   Channel explains less than 0.004% of ROAS variance (eta-squared = 0.00003870). Shifting ad budget across "
    f"   channels cannot be justified on historical campaign data alone.\n\n"
    f"2. UNIT ECONOMICS AND CONVERSIONS ARE NEARLY CONSTANT ACROSS CHANNELS:\n"
    f"   Estimated CAC ranges strictly between ${min_cac:.2f} (Website) and ${max_cac:.2f} (Google Ads), "
    f"   a difference of only $3.30 (1.17%). Cost per click spans $22.62 to $22.84.\n\n"
    f"3. NO ORGANIC TEMPORAL TRENDS OR BUDGET PENALTIES:\n"
    f"   Monthly revenue tracks calendar day counts perfectly (daily pacing is constant at ~$41.1M/day). "
    f"   Returns do not diminish at high acquisition costs (r = {cost_roas_corr:.4f} between cost and ROAS).\n\n"
    f"4. CUSTOMER INCOME IS STRONGLY ASSOCIATED WITH COMMERCIAL ENGAGEMENT:\n"
    f"   In the separate customer dataset (n=2,236), High-Income customers ($65k-$100k, n=662) average ${high_inc_spend:.2f} "
    f"   in spend and {high_inc_acc:.1f}% campaign acceptance, compared to ${low_inc_spend:.2f} and {low_inc_acc:.1f}% "
    f"   for Low-Income earners (<$35k, n=549)—a {spend_mult:.1f}x spend and {acc_mult:.1f}x acceptance advantage.\n\n"
    f"5. AGE EFFECT DISAPPEARS WHEN CONTROLLING FOR INCOME (HYPOTHESIS FOR TESTING):\n"
    f"   Cross-tabulation reveals that high earners spend ~$1,220-$1,430 across all age brackets. The agency should "
    f"   focus audience targeting on high-income segments rather than age cohorts alone."
)
print(takeaway_summary)

---
## Task 3: Report-Ready Channel Table & Bullet List
Generate `reports/stat_checks/report_tables.csv` and formatted ready-to-paste text bullets.

In [ ]:
total_portfolio_spend = df_camp['acquisition_cost'].sum()
total_portfolio_rev = df_camp['revenue'].sum()

report_rows = []
for ch_name in ['Email', 'Google Ads', 'Instagram', 'Facebook', 'Website', 'Youtube']:
    ch_sub = df_camp[df_camp['channel_used'] == ch_name]
    ch_sp = ch_sub['acquisition_cost'].sum()
    ch_rv = ch_sub['revenue'].sum()
    ch_conv = (ch_sub['clicks'] * ch_sub['conversion_rate']).sum()
    
    ch_cac = ch_sp / ch_conv
    ch_b_share = (ch_sp / total_portfolio_spend) * 100
    ch_r_share = (ch_rv / total_portfolio_rev) * 100
    
    report_rows.append({
        'channel': ch_name,
        'estimated_cac': round(ch_cac, 2),
        'budget_share_pct': round(ch_b_share, 2),
        'revenue_share_pct': round(ch_r_share, 2)
    })

df_report_table = pd.DataFrame(report_rows)
report_table_path = STAT_DIR / 'report_tables.csv'
df_report_table.to_csv(report_table_path, index=False)
print(f"Saved: {report_table_path.resolve()}\n")

print("=== Report-Ready Channel Table ===")
display(df_report_table)

print("\n=== Ready-to-Paste Finding 2 Bullet List ===")
for _, r in df_report_table.iterrows():
    print(f"- {r['channel']}: estimated CAC ${r['estimated_cac']:.2f}; budget share {r['budget_share_pct']:.2f}%; revenue share {r['revenue_share_pct']:.2f}%")